# 12. Frontend Streaming


## Learning Objectives

Learn how to stream LLM responses to users in real time.

This notebook covers:
- Understanding the basics of LangChain SDK streaming (`.stream()`, `.astream_events()`)
- Learning the structure and usage of the `useStream` React hook
- Understanding the `StreamEvent` protocol
- Consuming real-time streaming from the Python SDK
- Understanding real-time agent-state display patterns


## 12.1 Environment Setup


In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("환경 준비 완료.")

In [ ]:
# Optional observability setup: LangSmith or Langfuse
# Set the keys in .env, or uncomment the lines below to enter them manually.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: automatically enabled when LANGSMITH_TRACING=true
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: pass config={"callbacks": [langfuse_handler]} to invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 12.2 Python SDK Streaming Basics

The `.stream()` method delivers model output token by token in real time. Users can see partial results before the full response is complete.


In [ ]:
# .stream()으로 토큰 단위 스트리밍
print("스트리밍 응답:")
for chunk in model.stream(
    "서울의 유명한 관광지 3곳을 알려주세요.",
    config=lf_config,
):
    print(chunk.content, end="", flush=True)
print()  # 줄바꿈

## 12.3 `astream_events()`

`.astream_events()` streams **all internal events** asynchronously. This lets you trace model calls, tool execution, and chain steps in detail.

### Main event types

| Event | Description |
|--------|------|
| `on_chat_model_stream` | Model token streaming |
| `on_chat_model_start` | Model call started |
| `on_chat_model_end` | Model call completed |
| `on_tool_start` | Tool execution started |
| `on_tool_end` | Tool execution completed |


In [ ]:
import asyncio

async def stream_events_demo():
    """astream_events()로 이벤트 스트리밍 예시"""
    print("이벤트 스트리밍:")
    print("-" * 40)
    async for event in model.astream_events(
        "파이썬의 장점 2가지",
        version="v2",
    ):
        kind = event["event"]
        if kind == "on_chat_model_stream":
            content = event["data"]["chunk"].content
            if content:
                print(content, end="", flush=True)
        elif kind == "on_chat_model_start":
            print(f"[모델 호출 시작]")
        elif kind == "on_chat_model_end":
            print(f"\n[모델 호출 완료]")

await stream_events_demo()

## 12.4 The `useStream` React Hook

`useStream` is a React hook from the LangGraph SDK that simplifies streaming communication with a LangGraph server.

### Basic usage

```tsx
import { useStream } from "@langchain/langgraph-sdk/react";

function Chat() {
  const stream = useStream({
    assistantId: "agent",
    apiUrl: "http://localhost:2024",
  });

  const handleSubmit = (message: string) => {
    stream.submit({
      messages: [{ content: message, type: "human" }],
    });
  };

  return (
    <div>
      {stream.messages.map((message, idx) => (
        <div key={message.id ?? idx}>
          {message.type}: {message.content}
        </div>
      ))}
      {stream.isLoading && <div>Loading...</div>}
      {stream.error && <div>Error: {stream.error.message}</div>}
    </div>
  );
}
```

### Main return values

| Property | Type | Description |
|------|------|------|
| `messages` | `Message[]` | The full message list for the current thread |
| `isLoading` | `boolean` | Whether the stream is active |
| `error` | `Error \| null` | Error object |
| `interrupt` | `Interrupt` | An interruption request (HITL) |
| `submit()` | `function` | Send a message |
| `stop()` | `function` | Stop the stream |


## 12.5 `useStream` Configuration Options

| Parameter | Required | Default | Description |
|----------|------|--------|------|
| `assistantId` | O | — | Agent identifier (from the deployment dashboard) |
| `apiUrl` | — | `localhost:2024` | Agent server URL |
| `apiKey` | — | — | Auth token for a deployed agent |
| `threadId` | — | — | Connect to an existing conversation thread |
| `onThreadId` | — | — | Callback when a thread is created |
| `reconnectOnMount` | — | `false` | Reconnect to an active stream when the component mounts |
| `onCustomEvent` | — | — | Custom event handler |
| `onUpdateEvent` | — | — | State update handler |
| `onMetadataEvent` | — | — | Metadata event handler |
| `messagesKey` | — | `"messages"` | State key that stores messages |
| `throttle` | — | `true` | Batch state updates |
| `initialValues` | — | — | Cached initial state |


## 12.6 Thread Management and Reconnection

### Managing Thread IDs

By managing `threadId`, you can continue a conversation or load a previous thread.

```tsx
const [threadId, setThreadId] = useState<string | null>(null);

const stream = useStream({
  apiUrl: "http://localhost:2024",
  assistantId: "agent",
  threadId,
  onThreadId: setThreadId,
});

// Persist threadId in a URL parameter or localStorage
```

### Reconnecting After a Page Refresh

If you enable `reconnectOnMount`, the hook automatically reconnects to a stream that was already in progress after a page refresh.

```tsx
const stream = useStream({
  apiUrl: "http://localhost:2024",
  assistantId: "agent",
  reconnectOnMount: true, // use sessionStorage
});

// Use custom storage
const stream = useStream({
  reconnectOnMount: () => window.localStorage,
});
```


## 12.7 Branching and Message Editing

With branching, you can create an **alternate path** from a specific point in the conversation history. This is useful when you want to edit a user message or regenerate an AI response.

```tsx
{stream.messages.map((message) => {
  const meta = stream.getMessagesMetadata(message);
  const parentCheckpoint = meta?.firstSeenState?.parent_checkpoint;

  return (
    <div key={message.id}>
      {message.content}

      {/* Edit a user message */}
      {message.type === "human" && (
        <button onClick={() => {
          const newContent = prompt("Edit:", message.content);
          if (newContent) {
            stream.submit(
              { messages: [{ type: "human", content: newContent }] },
              { checkpoint: parentCheckpoint }
            );
          }
        }}>
          Edit
        </button>
      )}

      {/* Regenerate an AI response */}
      {message.type === "ai" && (
        <button onClick={() =>
          stream.submit(undefined, { checkpoint: parentCheckpoint })
        }>
          Regenerate
        </button>
      )}
    </div>
  );
})}
```

Key idea: use the `checkpoint` parameter to jump back to a specific state and generate a new branch.


## 12.8 Custom Streaming Events

You can stream **custom data** from the agent to the client. This is useful for progress updates, intermediate results, and other real-time signals.


In [ ]:
# 커스텀 스트리밍 이벤트 — Python writer 패턴
print("커스텀 스트리밍 이벤트 패턴 (Python 서버 측):")
print("=" * 50)
print("""
from langchain.tools import tool
from langchain.agents.types import ToolRuntime

@tool
async def analyze_data(
    data_source: str, *, config: ToolRuntime
) -> str:
    \"\"\"데이터를 분석합니다.\"\"\"
    if config.writer:
        # 진행 상황을 클라이언트에 스트리밍
        config.writer({
            "type": "progress",
            "message": "데이터 로딩 중...",
            "progress": 25,
        })
        # ... 처리 ...
        config.writer({
            "type": "progress",
            "message": "분석 완료!",
            "progress": 100,
        })
    return '{"result": "분석 완료"}'
""")
print("클라이언트(React) 측: onCustomEvent 콜백으로 수신")
print('  onCustomEvent: (data) => setProgress(data.progress)')

## 12.9 Multi-Agent Streaming

When several agents collaborate, their messages should be **displayed separately**. Use the `langgraph_node` metadata field to identify which agent produced each message.

### Event callback summary

| Callback | Use Case | Stream Mode |
|------|------|------------|
| `onUpdateEvent` | State update after a graph step | `updates` |
| `onCustomEvent` | Agent-defined custom event | `custom` |
| `onMetadataEvent` | Execution and thread metadata | `metadata` |
| `onError` | Error handling | — |
| `onFinish` | Stream completion | — |


## 12.10 Summary

This notebook covered:

| Topic | Key Idea |
|------|----------|
| **SDK streaming** | Use `.stream()` for real-time token output |
| **`astream_events`** | Trace model and tool calls through asynchronous event streaming |
| **`useStream`** | Simplify LangGraph server streaming in React |
| **Thread management** | Keep conversations alive with `threadId` and `reconnectOnMount` |
| **Branching** | Create alternate conversation paths from a `checkpoint` |
| **Custom events** | Stream progress and other custom data with a writer pattern |
| **Multi-agent** | Use `langgraph_node` metadata to distinguish messages by agent |

### Next Steps
→ Continue to **[13_guardrails.ipynb](./13_guardrails.ipynb)**


---
**References:**
- [Streaming](../docs/langchain/08-streaming.md)
- [UI (Agent Chat UI & useStream)](../docs/langchain/28-ui.md)
